# RBX-AI Studio en Google Colab (gratis, uso personal) — con ngrok

Este notebook monta tu servidor Node.js (RBX-AI Studio) en Colab y te da una URL pública con HTTPS para que tu plugin de Roblox Studio se conecte a ella.

**Requisito previo (1 minuto, solo la primera vez):**
1. Crea una cuenta gratis en [ngrok.com](https://dashboard.ngrok.com/signup)
2. Ve a [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) y copia tu **authtoken**.

**Cómo usar (3 pasos):**
1. Sube tu repo a GitHub.
2. En Colab: **File → Open notebook → GitHub** → pega la URL de tu repo y elige `colab_server.ipynb`.
3. Pulsa **Connect** (arriba a la derecha) y ejecuta la **Celda 1**. Te pedirá tu authtoken y tu `OPENROUTER_API_KEY` (ambos con `prompt`, nunca se guardan en el notebook). Cuando termine verás la URL en azul.

**Después:** abre la URL `https://....ngrok-free.app` en una pestaña nueva del navegador y pégala en la ventana **RBX-AI Bridge** del plugin en Roblox Studio.

**Nota:** la sesión dura hasta ~12 h. Al cerrarse, vuelve a ejecutar la Celda 1 (la URL cambia, actualízala en el plugin). Mantén la pestaña de Colab abierta mientras programas: el plugin hace polling cada 0.5 s.

In [ ]:
# Celda 1: descargar el proyecto, arrancar el servidor y crear el túnel ngrok
import os, subprocess, threading, time, re
import urllib.request
from google.colab import output

# ── 0. Authtoken de ngrok (cuenta gratis: dashboard.ngrok.com/get-started/your-authtoken) ──
authtoken = output.eval_js("prompt('Pega tu authtoken de ngrok (gratis en dashboard.ngrok.com/get-started/your-authtoken)')")
if not authtoken or authtoken == "null":
    raise RuntimeError("Sin authtoken no puedo crear el túnel. Consíguelo gratis en https://dashboard.ngrok.com/get-started/your-authtoken")

# ── 1. Clona tu repo de GitHub ───────────────────────────────────────────
REPO = "https://github.com/TU_USUARIO/TU_REPO.git"            # <--- TU REPO
FILES_PATH = "IA programacion roblox studio"                   # carpeta dentro del repo

os.chdir("/content")
if not os.path.exists("rbxai/server.js"):
    !git clone --depth 1 "$REPO" rbxai 2>/dev/null
    if not os.path.exists("rbxai/server.js"):
        os.makedirs("rbxai", exist_ok=True)
        # Fallback: descarga los archivos sueltos del repo
        base = "https://raw.githubusercontent.com/TU_USUARIO/TU_REPO/main/" + FILES_PATH + "/"
        for f in ["server.js", "index.html", "package.json", "package-lock.json", "rbxai_plugin.lua"]:
            !wget -q -O "rbxai/$f" "$base$f"

# ── 2. Tu API key de OpenRouter (la escribes tú, nunca se guarda) ────────
api_key = output.eval_js("prompt('Pega tu OPENROUTER_API_KEY de https://openrouter.ai/keys')")
if not api_key or api_key == "null":
    raise RuntimeError("Sin API key no puedo arrancar. Consíguela gratis en https://openrouter.ai/keys")
os.environ["OPENROUTER_API_KEY"] = api_key
PORT = "8080"
os.environ["PORT"] = PORT

# ── 3. Instalar dependencias y arrancar el servidor ──────────────────────
os.chdir("rbxai")
os.system("npm install --silent")

proc = subprocess.Popen(
    ["node", "server.js"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env=os.environ
)
def tail():
    for line in proc.stdout:
        print(line.decode(), end="")
threading.Thread(target=tail, daemon=True).start()
time.sleep(8)
print("✅ Servidor Node.js corriendo en el puerto " + PORT)

# ── 4. Instalar ngrok y registrar el authtoken ──────────────────────────
!curl -s -o /tmp/ngrok.zip -L "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip"
!unzip -o -q /tmp/ngrok.zip -d /usr/local/bin ngrok
!ngrok config add-authtoken "$authtoken" --quiet
print("✅ ngrok instalado y authtoken registrado")

# ── 5. Arrancar el túnel ────────────────────────────────────────────────
# --log=stdout: captura la URL fácilmente.
# Nota: las URLs gratuitas son https://XXXX.ngrok-free.app — sin pantalla de login.
tun = subprocess.Popen(
    ["ngrok", "http", PORT, "--log", "stdout"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
logfile = "/content/ngrok.log"
tun.stdout = open(logfile, "a")
tun.stderr = open(logfile, "a")

# ── 6. Capturar la URL pública (espera hasta 90 s) ──────────────────────
print("⏳ Creando túnel ngrok...")
url = None
for _ in range(30):
    time.sleep(3)
    try:
        log_text = open(logfile).read()
        m = re.search(r"https://[a-z0-9-]+\.ngrok-free\.app", log_text)
        if m:
            url = m.group(0)
            break
    except Exception:
        pass

# Verificar que la URL está viva antes de dártela
if url:
    ok = False
    for i in range(20):
        time.sleep(3)
        try:
            code = urllib.request.urlopen(url, timeout=15).getcode()
            if code == 200:
                ok = True
                break
        except Exception:
            pass
    print()
    print("=" * 64)
    if ok:
        print("🔗 TU URL PÚBLICA (ábrela en una pestaña NUEVA del navegador):")
        print("   " + url)
        print("   Pégala también en la ventana 'RBX-AI Bridge' del plugin en Roblox Studio")
        print("   (Mantén esta pestaña de Colab abierta mientras programas)")
    else:
        print("⚠️  El túnel arrancó pero la URL aún no responde. Revisa el log:")
        print(open(logfile).read()[-800:])
    print("=" * 64)
else:
    print("⚠️  No se pudo crear el túnel. Revisa el log de ngrok y verifica")
    print("   que el authtoken sea correcto. Detalle:")
    print(open(logfile).read()[-800:])